# Timing async vs sequential

Make sure the server is running.

In [ ]:
import asyncio, httpx, time

URL = 'http://localhost:8000/sleep/200'

async def seq(n):
    async with httpx.AsyncClient() as c:
        for _ in range(n):
            await c.get(URL)

async def conc(n):
    async with httpx.AsyncClient() as c:
        await asyncio.gather(*(c.get(URL) for _ in range(n)))

for n in (1, 5, 20):
    t = time.perf_counter(); asyncio.run(seq(n));  s = time.perf_counter()-t
    t = time.perf_counter(); asyncio.run(conc(n)); c = time.perf_counter()-t
    print(f'n={n:3}  seq={s*1000:7.1f} ms  conc={c*1000:7.1f} ms  speedup={s/c:.2f}x')

## Visualise it

Now demonstrate the blocking call freezes other requests.

In [ ]:
import threading, httpx, time

def hit_concurrent():
    t = time.perf_counter()
    r = httpx.get('http://localhost:8000/concurrent', timeout=10)
    print(f'concurrent finished in {(time.perf_counter()-t)*1000:.1f} ms')

def hit_blocking():
    t = time.perf_counter()
    r = httpx.get('http://localhost:8000/blocking-sleep', timeout=10)
    print(f'blocking-sleep finished in {(time.perf_counter()-t)*1000:.1f} ms')

th = threading.Thread(target=hit_blocking)
th.start()
time.sleep(0.05)
hit_concurrent()
th.join()